In [20]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade",
    "transformers>=4.41.0", "peft>=0.10.0", "trl>=0.8.6",
    "accelerate>=0.29.0", "datasets>=2.18.0",
    "evaluate>=0.4.0", "rouge_score", "nltk"],
    capture_output=True)
print("Installed")

Installed


In [21]:
import os, json, random, torch, requests, shutil, nltk
from kaggle_secrets import UserSecretsClient
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer
import evaluate as hf_eval
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HUGGINGFACE_HUB_TOKEN"] = HF_TOKEN
print("GPU:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))


GPU: True
GPU: Tesla P100-PCIE-16GB


In [22]:
random.seed(42)
found = []
for root, _, files in os.walk("/kaggle/input"):
    for f in files:
        if f.endswith(".jsonl"):
            found.append(os.path.join(root, f))
print("Found:", found)
DATASET_PATH = found[0]
with open('/kaggle/input/datasets/aditisikarwar/dataset-finetune/data_clean.jsonl') as f:
    raw = [json.loads(l) for l in f if l.strip()]
print(f"Examples: {len(raw)}")
print(f"Sample  : {raw[0]}")


Found: ['/kaggle/input/datasets/aditisikarwar/dataset-finetune/data_clean.jsonl']
Examples: 799
Sample  : {'input': "You're overreacting, end of discussion.", 'output': 'Let me see how I can help. I understand this has caused you frustration, and I want you to know your concerns are completely valid.'}


In [23]:
INSTRUCTION = (
    "Rewrite the following blunt customer support reply "
    "into a warm, professional response. Keep the core message intact."
)
def fmt(ex):
    return {
        "text": f"### Instruction:\n{INSTRUCTION}\n\n### Input:\n{ex['input']}\n\n### Response:\n{ex['output']}",
        "input_raw": ex["input"],
        "output_raw": ex["output"],
    }
ds    = Dataset.from_list([fmt(e) for e in raw])
split = ds.train_test_split(test_size=0.1, seed=42)
train_ds, val_ds = split["train"], split["test"]
print(f"Train: {len(train_ds)} | Val: {len(val_ds)}")

Train: 719 | Val: 80


In [24]:
MODEL_ID = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(MODEL_ID)  # CPU, no cuda
model.config.use_cache = False
print("Model loaded")
print(f"VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB")
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torchao>=0.16.0"])
import importlib, peft
importlib.reload(peft)
from peft import LoraConfig, get_peft_model, TaskType
print("done")

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Model loaded
VRAM used: 0.00 GB
done


In [25]:
lora_cfg = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    target_modules=["c_attn"],
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()
print("Lora applied")
import trl, inspect
print(trl.__version__)
print(inspect.signature(trl.SFTConfig.__init__))

trainable params: 147,456 || all params: 82,060,032 || trainable%: 0.1797
Lora applied
0.8.6
(self, output_dir: str | None = None, per_device_train_batch_size: int = 8, num_train_epochs: float = 3.0, max_steps: int = -1, learning_rate: float = 2e-05, lr_scheduler_type: transformers.trainer_utils.SchedulerType | str = 'linear', lr_scheduler_kwargs: dict | str | None = None, warmup_steps: float = 0, optim: transformers.training_args.OptimizerNames | str = 'adamw_torch_fused', optim_args: str | None = None, weight_decay: float = 0.0, adam_beta1: float = 0.9, adam_beta2: float = 0.999, adam_epsilon: float = 1e-08, optim_target_modules: None | str | list[str] = None, gradient_accumulation_steps: int = 1, average_tokens_across_devices: bool = True, max_grad_norm: float = 1.0, label_smoothing_factor: float = 0.0, bf16: bool | None = None, fp16: bool = False, bf16_full_eval: bool = False, fp16_full_eval: bool = False, tf32: bool | None = None, gradient_checkpointing: bool = True, gradient_chec

/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/layer.py:2504: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


In [26]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""  
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model=model,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=SFTConfig(
        output_dir="/kaggle/working/lora-distilgpt2-support",
        num_train_epochs=3,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        gradient_accumulation_steps=2,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        learning_rate=2e-4,
        fp16=False,
        bf16=False,
        use_cpu=True,
        logging_steps=10,
        report_to="none",
        max_length=128,
        dataset_text_field="text",
        packing=False,
    ),
)
print("Training")
trainer.train()
print("Done")
trainer.model.save_pretrained("/kaggle/working/lora-distilgpt2-support")

Adding EOS to train dataset:   0%|          | 0/719 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/719 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/80 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/80 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.


Training


Epoch,Training Loss,Validation Loss
1,2.471829,2.137749
2,1.804908,1.607679
3,1.737440,1.574033


Done


In [27]:
model.eval()
def rewrite(blunt, max_new_tokens=120):
    prompt = f"# Instruction:\n{INSTRUCTION}\n\n# Input:\n{blunt}\n\n# Response:\n"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LEN)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.4,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1,
        )
    new = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new, skip_special_tokens=True).strip()
print("Inference ready")

Inference ready


In [28]:
DEMO = [
    "We can't help you with this.",
    "Not our problem. Call the carrier.",
    "Stop emailing us so much.",
    "You entered the wrong details. Try again.",
    "We don't see any issue here.",
]

print("=" * 65)
print("  BEFORE vs AFTER")
print("=" * 65)

qual_results = []
for i, blunt in enumerate(DEMO, 1):
    polite = rewrite(blunt)
    qual_results.append({"input": blunt, "output": polite})
    print(f"\n[{i}] BEFORE: {blunt}")
    print(f"    AFTER : {polite}")

  BEFORE vs AFTER

[1] BEFORE: We can't help you with this.
    AFTER : This is not your fault and we apologize for any inconvenience that may have occurred to us or our customers in advance of contacting you today.

[2] BEFORE: Not our problem. Call the carrier.
    AFTER : We can't help but feel that we need to address this issue in an appropriate way and do everything within our power to resolve it quickly.

[3] BEFORE: Stop emailing us so much.
    AFTER : I'm sorry about that and I apologize for any inconvenience you may have caused to me in this situation.

[4] BEFORE: You entered the wrong details. Try again.
    AFTER : I apologize for this inconvenience and I can't help but feel that you are being held responsible by me as well.

[5] BEFORE: We don't see any issue here.
    AFTER : Thank you for your patience and understanding.


In [29]:
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
rouge = hf_eval.load("rouge")
N = min(60, len(val_ds))
print(f"Evaluating {N} examples...")
refs, hyps = [], []
for i, ex in enumerate(val_ds.select(range(N))):
    hyps.append(rewrite(ex["input_raw"]))
    refs.append(ex["output_raw"])
    if (i+1) % 10 == 0:
        print(f"  {i+1}/{N}")
rouge_scores = rouge.compute(predictions=hyps, references=refs)
print("\n ROUGE")
for k, v in rouge_scores.items():
    print(f"  {k}: {v:.4f}")

bleu = corpus_bleu(
    [[nltk.word_tokenize(r.lower())] for r in refs],
    [nltk.word_tokenize(h.lower()) for h in hyps],
    smoothing_function=SmoothingFunction().method1
)
print(f"\nBLEU")
print(f"  {bleu:.4f}")

Evaluating 60 examples...
  10/60
  20/60
  30/60
  40/60
  50/60
  60/60

 ROUGE
  rouge1: 0.2229
  rouge2: 0.0349
  rougeL: 0.1678
  rougeLsum: 0.1688

BLEU
  0.0108


In [31]:
with open("/kaggle/working/qualitative_results.json", "w") as f:
    json.dump(qual_results, f, indent=2)
with open("/kaggle/working/eval_report.json", "w") as f:
    json.dump({
        "model": MODEL_ID,
        "train_size": len(train_ds),
        "val_size": len(val_ds),
        "rouge": rouge_scores,
        "bleu": bleu,
        "llm_judge": judge_results,
    }, f, indent=2)
shutil.make_archive("/kaggle/working/lora_adapter", "zip", OUTPUT_DIR)
print("Saved to /kaggle/working/")

Saved to /kaggle/working/
